In [ ]:
# =============================================
# TUGAS 4 - SISTEM TEMU KEMBALI INFORMASI
# PEMBOBOTAN (TERM WEIGHTING) DENGAN TF-IDF
# Nama: ABID SABRI ZAKI
# NIM : 240210502022
# =============================================

import re
import math
import pandas as pd
import numpy as np

# Install Sastrawi jika belum tersedia
!pip install Sastrawi

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer

# Setup Sastrawi
stemmer = StemmerFactory().create_stemmer()
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

print("Library berhasil di-import.")


In [ ]:
# Dataset: 4 dokumen pendek berbahasa Indonesia
# Topik dibuat berbeda dari contoh agar menjadi notebook baru.

dokumen = [
    "Teknologi informasi membantu manusia mengolah data dengan cepat dan akurat.",
    "Basis data digunakan untuk menyimpan dan mengelola informasi secara terstruktur.",
    "Jaringan komputer memungkinkan perangkat saling terhubung untuk bertukar data.",
    "Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan."
]

print("Dataset Dokumen:")
for i, doc in enumerate(dokumen, 1):
    print(f"Dok {i}: {doc}")


In [ ]:
# Preprocessing sederhana:
# 1. Mengubah huruf menjadi lowercase
# 2. Menghapus tanda baca
# 3. Menghapus stopword
# 4. Memisahkan teks menjadi token

def preprocess_sederhana(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = stopword_remover.remove(text)
    tokens = text.split()
    return tokens

tokens_dokumen = [preprocess_sederhana(doc) for doc in dokumen]

print("Hasil Preprocessing (Token):")
for i, tokens in enumerate(tokens_dokumen, 1):
    print(f"Dok {i}: {tokens}")


In [ ]:
# Bangun vocabulary (semua term unik dari seluruh dokumen)
vocab = sorted(set(term for tokens in tokens_dokumen for term in tokens))

print("Vocabulary:")
print(vocab)
print("Jumlah term unik:", len(vocab))

# Bangun matriks Bag-of-Words (raw count)
bow_matrix = []

for tokens in tokens_dokumen:
    row = [tokens.count(term) for term in vocab]
    bow_matrix.append(row)

df_bow = pd.DataFrame(
    bow_matrix,
    columns=vocab,
    index=[f'Dok {i+1}' for i in range(len(dokumen))]
)

print("\nMatriks Bag-of-Words (Raw Count):")
print(df_bow)


In [ ]:
# Term Frequency (TF)
# Pada contoh ini TF menggunakan raw count,
# sehingga nilainya sama dengan matriks Bag-of-Words.

df_tf = df_bow.copy()

print("Term Frequency (TF) Manual:")
print(df_tf)


In [ ]:
# Document Frequency (DF)
# DF = jumlah dokumen yang mengandung suatu term

N = len(dokumen)
df = {}

for term in vocab:
    df[term] = sum(1 for tokens in tokens_dokumen if term in tokens)

# IDF = log(N / df)
idf = {
    term: math.log(N / df[term])
    for term in vocab
}

df_idf = pd.DataFrame({
    'Term': vocab,
    'df': [df[t] for t in vocab],
    'IDF = log(N/df)': [round(idf[t], 4) for t in vocab]
})

print(f"Jumlah dokumen (N) = {N}")
print("\nTabel DF dan IDF:")
print(df_idf.to_string(index=False))


In [ ]:
# TF-IDF Manual
# Rumus:
# TF-IDF = TF × IDF

tfidf_manual = []

for i, tokens in enumerate(tokens_dokumen):
    row = []

    for term in vocab:
        tf_val = df_tf.loc[f'Dok {i+1}', term]
        idf_val = idf[term]
        row.append(tf_val * idf_val)

    tfidf_manual.append(row)

df_tfidf_manual = pd.DataFrame(
    tfidf_manual,
    columns=vocab,
    index=[f'Dok {i+1}' for i in range(len(dokumen))]
)

print("Matriks TF-IDF (Manual):")
print(df_tfidf_manual.round(4))


In [ ]:
# Implementasi TF-IDF menggunakan Scikit-learn
# Teks hasil preprocessing digabungkan kembali menjadi string.

teks_preprocessed = [' '.join(tokens) for tokens in tokens_dokumen]

vectorizer = TfidfVectorizer()
tfidf_sklearn = vectorizer.fit_transform(teks_preprocessed)

print("Vocabulary Scikit-learn:")
print(vectorizer.get_feature_names_out())

df_tfidf_sklearn = pd.DataFrame(
    tfidf_sklearn.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f'Dok {i+1}' for i in range(len(dokumen))]
)

print("\nMatriks TF-IDF (Scikit-learn):")
print(df_tfidf_sklearn.round(4))


In [ ]:
# Perbandingan dimensi matriks

print("Perbandingan Dimensi Matriks:")
print(f"Manual      : {df_tfidf_manual.shape}")
print(f"Scikit-learn: {df_tfidf_sklearn.shape}")

print("\nCatatan Perbedaan:")
print("- Perhitungan manual menggunakan IDF = log(N/df).")
print("- Scikit-learn menggunakan smooth IDF = log((1+N)/(1+df)) + 1.")
print("- Scikit-learn melakukan normalisasi L2 pada setiap vektor dokumen.")
print("- Karena formula dan normalisasi berbeda, nilai TF-IDF tidak harus sama.")


In [ ]:
# Analisis term dengan bobot TF-IDF tertinggi per dokumen
print("TERM DENGAN BOBOT TF-IDF TERTINGGI PER DOKUMEN")
print("=" * 60)

for i in range(len(dokumen)):
    row = df_tfidf_manual.loc[f'Dok {i+1}']
    top_terms = row[row > 0].sort_values(ascending=False).head(3)

    print(f"\nDokumen {i+1}:")
    for term, val in top_terms.items():
        print(f"  - {term}: {val:.4f}")


In [ ]:
# Menyajikan top-3 term dalam DataFrame

top_terms_per_doc = {}

for i in range(len(dokumen)):
    row = df_tfidf_manual.loc[f'Dok {i+1}']
    top = row[row > 0].sort_values(ascending=False).head(3)
    top_terms_per_doc[f'Dok {i+1}'] = top

data_top = []

for doc, top in top_terms_per_doc.items():
    for rank, (term, val) in enumerate(top.items(), 1):
        data_top.append({
            'Dokumen': doc,
            'Peringkat': rank,
            'Term': term,
            'Bobot TF-IDF': round(val, 4)
        })

df_top_terms = pd.DataFrame(data_top)

print(df_top_terms.to_string(index=False))


In [ ]:
# =============================================
# ANALISIS TERM DENGAN BOBOT TF-IDF TERTINGGI
# =============================================

analisis_poin = '''
1. DOKUMEN 1 — Topik: Teknologi Informasi
   Term yang memiliki bobot tinggi merupakan kata yang relatif spesifik
   terhadap isi dokumen. Kata yang hanya muncul pada dokumen tertentu
   memperoleh nilai IDF yang lebih tinggi.

2. DOKUMEN 2 — Topik: Basis Data
   Term yang berkaitan dengan penyimpanan, pengelolaan, dan struktur data
   dapat menjadi pembeda dokumen karena tidak muncul secara merata pada
   seluruh koleksi dokumen.

3. DOKUMEN 3 — Topik: Jaringan Komputer
   Term yang berkaitan dengan perangkat, jaringan, dan hubungan antarperangkat
   mempunyai kemampuan membedakan dokumen ini dari dokumen lain.

4. DOKUMEN 4 — Topik: Sistem Temu Kembali Informasi
   Term yang berkaitan dengan pencarian, dokumen, informasi, dan relevansi
   menjadi ciri penting karena berhubungan langsung dengan topik dokumen.

KESIMPULAN:
TF-IDF memberikan bobot lebih besar pada term yang muncul dalam dokumen
tetapi jarang ditemukan pada dokumen lain. Sebaliknya, term yang muncul
di banyak dokumen mempunyai nilai IDF lebih rendah sehingga kontribusinya
terhadap pembeda antar dokumen menjadi lebih kecil.
'''

print(analisis_poin)


In [ ]:
# =============================================
# ANALISIS SINGKAT
# =============================================

Analisis_singkat = '''
Berdasarkan perhitungan TF-IDF secara manual dan menggunakan Scikit-learn,
dapat dilihat bahwa term yang lebih spesifik terhadap suatu dokumen cenderung
memiliki bobot yang lebih tinggi. Perhitungan manual menggunakan rumus
TF-IDF = TF × IDF dengan IDF = log(N/df), sedangkan Scikit-learn menggunakan
smooth IDF dan normalisasi L2 sehingga nilai akhirnya dapat berbeda.

Perbedaan tersebut tidak berarti prosesnya salah, karena metode perhitungan
yang digunakan memang berbeda. Prinsip utamanya tetap sama, yaitu menekan
term yang terlalu umum dan memberikan bobot lebih besar kepada term yang
lebih jarang muncul pada dokumen lain.

Dengan demikian, TF-IDF dapat digunakan sebagai salah satu metode pembobotan
dalam sistem temu kembali informasi untuk membantu menentukan term yang
lebih representatif pada setiap dokumen.
'''

print(Analisis_singkat)
